# Lesson 4: Managing Context

**The central challenge:** LLMs are stateless functions. They don't remember anything — you must send the relevant context with every request. This has profound implications for cost, latency, and architecture.

This notebook walks through five strategies, each solving a problem the previous one creates:

| # | Strategy | Problem it solves | New problem it creates |
|---|----------|-------------------|----------------------|
| 1 | **Stateless** | Simplest possible call | No memory at all |
| 2 | **Stateful (full history)** | Model remembers everything | Quadratic cost growth |
| 3 | **Sliding window** | Bounded cost | Forgets old context |
| 4 | **Window + summarization** | Preserves old context cheaply | Lossy compression |
| 5 | **Long-term memory** | Persists across sessions | Needs user identity + storage |

Run cells in order. Each section builds on the previous one.

In [ ]:
import os, time, json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found — add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
print(f"Client ready — using {MODEL}")

---
## 1) Stateless: each call is independent

The Chat Completions API is **stateless by design**. There is no server-side session. The model only sees the `messages` array you send in *that* request — nothing more.

Below we make two separate calls. Turn 1 tells the model a name; turn 2 asks for it. Since turn 2 doesn't include turn 1's messages, the model has no idea.

In [ ]:
# Turn 1: tell the model your name
turn1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Alice."}],
)
print("Turn 1:", turn1.choices[0].message.content)

# Turn 2: ask for your name — but we DON'T send turn 1
turn2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("Turn 2:", turn2.choices[0].message.content)

The model can't answer because it literally never saw the first message. Each call is a clean slate.

**When stateless is fine:** single-shot tasks (translation, classification, extraction) where each request is self-contained. Cost per call is constant and predictable.

---
## 2) Stateful: replay the full history

The fix is simple: keep a `messages` list and append every exchange. On each turn, send the whole thing. This is exactly how ChatGPT works under the hood — the "memory" lives in the client, not on the server.

In [ ]:
history = []

# Turn 1
history.append({"role": "user", "content": "My name is Alice."})

r1 = client.chat.completions.create(model=MODEL, messages=history)
a1 = r1.choices[0].message.content
history.append({"role": "assistant", "content": a1})
print("Turn 1:", a1)


# Turn 2 — now the model sees BOTH messages
history.append({"role": "user", "content": "What is my name?"})
r2 = client.chat.completions.create(model=MODEL, messages=history)
a2 = r2.choices[0].message.content
history.append({"role": "assistant", "content": a2})
print("Turn 2:", a2)

print(f"\nMessages sent on turn 2: {len(history) - 1}")  # minus the last assistant reply
print(f"Prompt tokens on turn 2: {r2.usage.prompt_tokens}")

It works — but notice: on turn 2, we sent *all* previous messages plus the new one. On turn 50, we'd send 50 turns of history. On turn 500, we'd send 500 turns. This is where cost becomes a problem.

---
## 3) Measuring the quadratic problem

Let's run a real multi-turn conversation and watch the token count grow. If each turn adds ~300 tokens to the history, the input tokens on turn N are roughly 300N. The **cumulative** input tokens across all turns grow as ~150N² — quadratic.

In [ ]:
prompts = [
    "My name is Alice and I live in Trento.",
    "I work as a data scientist at a small startup.",
    "I prefer concise answers — no fluff, just the key points.",
    "I'm building a recommendation engine using collaborative filtering.",
    "We're using PyTorch and the MovieLens 1M dataset.",
    "What RMSE should I target for a good baseline?",
    "Summarize everything you know about me so far.",
    "What details might be missing from my project description?",
    "Give me three concrete next steps for my project.",
    "What is my name, where do I live, and what am I working on?",
    "how are you today? i am fine",
    "what shall I do today? i have to do some work",
]

history = []
stats = []

for i, prompt in enumerate(prompts, start=1):
    history.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=history, temperature=0.2)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    print(f"\n--- Turn {i} ---\nUser: {prompt}\nAssistant: {reply}\n")

    stats.append({
        "turn": i,
        "prompt_tokens": r.usage.prompt_tokens,
        "completion_tokens": r.usage.completion_tokens,
    })

# Print the growth table
print(f"{'turn':>4}  {'prompt_tokens':>14}  {'completion_tokens':>18}  {'cumulative_input':>17}")
cumulative = 0
for s in stats:
    cumulative += s["prompt_tokens"]
    print(f"{s['turn']:>4}  {s['prompt_tokens']:>14,}  {s['completion_tokens']:>18,}  {cumulative:>17,}")

Notice how `prompt_tokens` grows with every turn — each turn re-sends all previous turns. The last turn pays for the entire conversation.

**The cost math:** at gpt-4.1-mini pricing ($0.40/M input tokens), a 10-turn conversation is pennies. But scale to 500 turns and you're at ~$15 cumulative input cost for a single conversation. The model choice matters too — cheaper models (gpt-4.1-nano at $0.10/M) reduce cost 4×.

**Latency grows too:** more input tokens = more processing time. By turn 100+, users may wait 5-10 seconds per response.

---
## 4) Sliding window: bounded cost, lossy context

The simplest fix: only send the last N turns. Cost is now **bounded** regardless of conversation length.

In [ ]:
def trim_to_window(conversation, window_turns=3):
    """Keep only the last `window_turns` exchanges (user + assistant pairs).
    
    Counts user messages from the end to find turn boundaries.
    """
    if window_turns <= 0:
        return []
    kept = []
    user_count = 0
    for msg in reversed(conversation):
        kept.append(msg)
        if msg["role"] == "user":
            user_count += 1
            if user_count >= window_turns:
                break
    return list(reversed(kept))

In [ ]:
# Demo: window of 2 turns on a 10-turn conversation
full_history = history  # from section 3
window = trim_to_window(full_history, window_turns=2)

print(f"Full history: {len(full_history)} messages ({len(full_history)//2} turns)")
print(f"Window (last 2 turns): {len(window)} messages\n")

for msg in window:
    role = msg["role"].upper()
    preview = msg["content"][:100].replace("\n", " ")
    print(f"  [{role}] {preview}{'...' if len(msg['content']) > 100 else ''}")

In [ ]:
# Now ask a question that REQUIRES early context
# With a small window, the model won't know Alice's name or project

windowed_messages = window + [{"role": "user", "content": "What is my name and what project am I working on?"}]

r = client.chat.completions.create(model=MODEL, messages=windowed_messages, temperature=0.2)
print("With window=2 (early context lost):")
print(r.choices[0].message.content)
print(f"\nPrompt tokens: {r.usage.prompt_tokens} (bounded!)")

The model can't answer — Alice's name and project details were in turns 1-5, outside the window. Cost is bounded, but we've lost important context.

**The tradeoff triangle — pick two:**
- Full context (model sees everything, best quality)
- Low cost (minimal tokens per turn)
- Low latency (fast responses)

We need a way to *compress* old context, not just drop it.

---
## 5) Answer + memory in one call

Instead of a separate summarization step, we can ask the model to do both in a single call:
1. Answer the user's question
2. Update a running memory summary

Each turn sends: `[system prompt with current memory] + [user message]` → model returns `{"answer": "...", "updated_memory": {...}}`

The memory is a structured JSON dict that grows and evolves across turns. Cost is **bounded** — we never send the full history, just the compact memory.

In [ ]:
from prompt_templates import render_prompt


def chat_with_memory(user_message, memory=None):
    """Send a message, get an answer and updated memory in one call.

    Args:
        user_message: the user's input
        memory: dict with facts/preferences/goals (or None for first turn)

    Returns:
        (answer, updated_memory, usage) tuple
    """
    system, user = render_prompt(
        "summarize_memory.j2",
        memory=json.dumps(memory) if memory else "",
        user_message=user_message,
    )
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.3,
        response_format={"type": "json_object"},
    )
    parsed = json.loads(r.choices[0].message.content)
    return (
        parsed.get("answer", ""),
        parsed.get("updated_memory", memory or {}),
        r.usage,
    )


print("chat_with_memory() defined.")

In [ ]:
# Single turn demo
answer, memory, usage = chat_with_memory("Hi! My name is Alice and I live in Trento.")

print(f"Answer: {answer}")
print(f"Memory: {json.dumps(memory, indent=2)}")
print(f"Prompt tokens: {usage.prompt_tokens}")

In [ ]:
# Second turn — memory carries over, no history needed
answer, memory, usage = chat_with_memory(
    "I work as a data scientist. I prefer concise answers.", memory
)
print(f"Answer: {answer}")
print(f"Memory: {json.dumps(memory, indent=2)}")
print(f"Prompt tokens: {usage.prompt_tokens}")

# Third turn — test recall
answer, memory, usage = chat_with_memory(
    "What is my name and what do I do?", memory
)
print(f"\nAnswer: {answer}")
print(f"Prompt tokens: {usage.prompt_tokens} (bounded!)")

The model answers correctly using only the compact memory — no conversation history needed.

**How it works:** each turn, the model sees only the system prompt (with memory JSON) and the current user message. The memory acts as a lossy compression of the entire conversation so far.

```
[SYSTEM]  You are a helpful assistant.
          CURRENT MEMORY: {"facts": ["Name: Alice", "Location: Trento", ...], ...}
          ... return {"answer": "...", "updated_memory": {...}} ...

[USER]    What is my name and what do I do?    ← only the current turn
```

One API call per turn. Bounded tokens. No growing history.

---
## 6) Multi-turn conversation with memory

Let's run a longer conversation and watch the memory evolve.

In [ ]:
prompts = [
    "Hi! I'm Marco, a PhD student in Trento studying graph neural networks.",
    "I'm applying GNNs to drug discovery — specifically protein-ligand binding prediction.",
    "I use PyTorch Geometric and I prefer short, direct explanations.",
    "What are the main challenges in GNN-based drug discovery?",
    "How does message passing relate to molecular graphs?",
    "Can you explain virtual nodes and why they help?",
    "What evaluation metrics should I use for binding affinity prediction?",
    "Remind me: what's my research topic and what framework do I use?",
]

memory = None
turn_stats = []

for i, prompt in enumerate(prompts, start=1):
    answer, memory, usage = chat_with_memory(prompt, memory)

    turn_stats.append({
        "turn": i,
        "prompt_tokens": usage.prompt_tokens,
        "memory_size": len(json.dumps(memory)),
    })
    print(f"--- Turn {i} ({usage.prompt_tokens} prompt tokens) ---")
    print(f"You:  {prompt}")
    print(f"Asst: {answer[:200]}{'...' if len(answer) > 200 else ''}\n")

# Token count stays bounded
print("\n=== Token growth (bounded!) ===")
print(f"{'turn':>4}  {'prompt_tokens':>14}  {'memory_json':>12}")
for s in turn_stats:
    print(f"{s['turn']:>4}  {s['prompt_tokens']:>14,}  {s['memory_size']:>12,}")

In [ ]:
# Show the final memory
print("=== Final memory ===")
print(json.dumps(memory, indent=2))

Key observation: the `prompt_tokens` column stays roughly bounded even as the conversation grows. The memory summary grows slowly (it gets compressed further on each update), while the window stays fixed. Compare this to section 3 where tokens grew linearly per turn and quadratically in total.

---
## 7) Long-term memory: persisting across sessions

Everything above is **short-term memory** — it dies when the conversation ends. Real applications need to remember users *across* conversations.

The pattern:
1. **End of session:** ask the LLM to extract structured facts from the conversation
2. **Save** those facts to disk (or a database), keyed by user ID
3. **Start of next session:** load the user's facts and inject them into the system prompt

This is the pattern behind ChatGPT's "Memory" feature. It requires **user identity** — you need to know *whose* facts to load.

In [ ]:
import tempfile

def extract_long_term_facts(conversation, existing_memory=None):
    """Ask the LLM to extract facts worth remembering across sessions.
    
    The LLM itself decides what's important enough to persist:
    facts, preferences, expertise, decisions — not ephemeral details.
    """
    if existing_memory is None:
        existing_memory = {"facts": [], "preferences": []}

    transcript = "\n".join(
        f"{m['role'].title()}: {m['content'][:300]}" for m in conversation
    )
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Extract LONG-TERM memory from this conversation.\n"
                "Include: facts about the user, preferences, decisions, expertise.\n"
                "Exclude: temporary details, one-off questions, ephemeral context.\n"
                "Return valid JSON: {\"facts\": [...], \"preferences\": [...]}\n"
                "Max 10 items per list. Merge with existing memory, remove duplicates."
            )},
            {"role": "user", "content": (
                f"EXISTING MEMORY:\n{json.dumps(existing_memory)}\n\n"
                f"CONVERSATION:\n{transcript}"
            )},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    try:
        return json.loads(r.choices[0].message.content)
    except json.JSONDecodeError:
        return existing_memory


# Use a temp file so we don't leave artifacts
memory_file = Path(tempfile.mkdtemp()) / "user_memories.json"


def save_user_memory(user_id, memory_data):
    """Save long-term memory for a user to disk."""
    all_memories = json.loads(memory_file.read_text()) if memory_file.exists() else {}
    all_memories[user_id.lower()] = memory_data
    memory_file.write_text(json.dumps(all_memories, indent=2))


def load_user_memory(user_id):
    """Load long-term memory for a user from disk."""
    if memory_file.exists():
        all_memories = json.loads(memory_file.read_text())
        return all_memories.get(user_id.lower(), {"facts": [], "preferences": []})
    return {"facts": [], "preferences": []}

In [ ]:
# ── SESSION 1: Marco has a conversation ──────────────────────────

session1_conversation = [
    {"role": "user", "content": "Hi, I'm Marco. I'm a PhD student in Trento."},
    {"role": "assistant", "content": "Nice to meet you, Marco! What are you working on?"},
    {"role": "user", "content": "Graph neural networks for drug discovery — protein-ligand binding."},
    {"role": "assistant", "content": "Fascinating area! What tools are you using?"},
    {"role": "user", "content": "PyTorch Geometric mostly. And I prefer short, direct explanations."},
    {"role": "assistant", "content": "Noted — I'll keep things concise. How's the research going?"},
]

print("SESSION 1: Extracting long-term facts...")
extracted = extract_long_term_facts(session1_conversation)
save_user_memory("marco", extracted)

print(f"  Facts:       {extracted.get('facts', [])}")
print(f"  Preferences: {extracted.get('preferences', [])}")
print(f"  Saved to: {memory_file}\n")

# ── SESSION 2: New conversation — load memory ───────────────────

print("SESSION 2: Loading memory for 'marco'...")
loaded = load_user_memory("marco")
print(f"  Loaded: {json.dumps(loaded, indent=2)}\n")

# Use the loaded memory in a new conversation
session2_messages = [
    {"role": "system", "content": (
        "You are a helpful assistant.\n\n"
        f"LONG-TERM MEMORY (from previous sessions):\n"
        f"Facts: {loaded.get('facts', [])}\n"
        f"Preferences: {loaded.get('preferences', [])}\n\n"
        "Use this memory naturally — don't say 'according to my records'."
    )},
    {"role": "user", "content": "What do you know about me?"},
]

r = client.chat.completions.create(model=MODEL, messages=session2_messages, temperature=0.3)
print("Marco asks: 'What do you know about me?'")
print(f"Assistant: {r.choices[0].message.content}")

# Clean up temp file
memory_file.unlink(missing_ok=True)
memory_file.parent.rmdir()

The model "remembers" Marco from a previous session it never actually saw — because we loaded his facts into the system prompt.

**Privacy note:** long-term memory means you're storing personal data. In production you need: clear data policies, user controls (view/delete), secure storage, and compliance with regulations (GDPR etc.). Our JSON file is fine for learning, not for production.

| Product | Short-term memory | Long-term memory |
|---------|-------------------|------------------|
| ChatGPT | Conversation history | "Memory" feature (extracted facts) |
| Claude | Conversation context | Project Knowledge (uploaded docs) |
| GitHub Copilot | Current file + open tabs | Repository patterns, coding style |
| Customer service bot | Current ticket | Customer history, past issues |

---
## 8) Exercises

Try these to deepen your understanding.

### Exercise 1: Cost calculator

Write a function `estimate_cost(n_turns, strategy)` that estimates total input tokens for an N-turn conversation under three strategies: `"full_history"`, `"windowed"` (window=6), and `"windowed_summary"` (window=6, summary ~200 tokens).

Assume 100 tokens/user message and 200 tokens/assistant response.

In [ ]:
def estimate_cost(n_turns, strategy="full_history"):
    """Estimate total input tokens over an N-turn conversation.
    
    Args:
        n_turns: number of conversation turns
        strategy: "full_history", "windowed", or "windowed_summary"
    
    Returns:
        Total input tokens across all turns (cumulative)
    """
    USER_TOKENS = 100
    ASST_TOKENS = 200
    TURN_TOKENS = USER_TOKENS + ASST_TOKENS  # 300 tokens per turn
    WINDOW = 6
    SUMMARY_TOKENS = 200

    # TODO: implement the three strategies
    # Hint for full_history: turn k sends k*TURN_TOKENS - ASST_TOKENS tokens
    # Hint for windowed: each turn sends min(k, WINDOW) * TURN_TOKENS tokens
    # Hint for windowed_summary: like windowed, but add SUMMARY_TOKENS when k > WINDOW
    pass


# Test it:
# for strategy in ["full_history", "windowed", "windowed_summary"]:
#     tokens = estimate_cost(50, strategy)
#     cost = tokens * 0.40 / 1_000_000  # gpt-4.1-mini pricing
#     print(f"{strategy:>20}: {tokens:>10,} tokens  ${cost:.4f}")

### Exercise 2: Window size experiment

Using `trim_to_window` and `build_messages_with_memory` (without summarization — just the window), run the same 10-prompt conversation from section 3 with window sizes of 1, 3, and 8. On the last turn, ask "What is my name, where do I live, and what am I working on?" For each window size, record whether the model gets it right. At what window size does it start failing?

In [ ]:
# TODO: for each window_size in [1, 3, 8]:
#   1. Take full_history (from section 3)
#   2. Trim to window using trim_to_window(full_history, window_size)
#   3. Build messages with build_messages_with_memory("", window, test_question)
#   4. Call the API and print the result
#   5. Check: does it know Alice's name? her location? her project?

test_question = "What is my name, where do I live, and what am I working on?"

# Your code here

### Exercise 3: Memory format comparison

The `chat_with_memory` function stores memory as `{"facts": [...], "preferences": [...], "goals": [...]}`. Try modifying the template to use two other formats:

1. **Flat list** — `{"memory": ["Name: Alice", "Lives in Trento", "Prefers concise answers"]}`
2. **Paragraph** — `{"memory": "Alice is a data scientist in Trento who prefers concise answers..."}`

Run the same 8-turn Marco conversation (section 6) with each format. Compare:
- Which preserves the most information after 8 turns?
- Which is most compact (fewest tokens)?
- Which produces the best answers?

In [ ]:
# TODO: create summarize_turns_json() and summarize_turns_paragraph() variants
# then run the section 6 conversation with each and compare results

# Your code here

### Exercise 4: Token-based windowing

Our `trim_to_window` counts *turns*. But turns vary wildly in length — a 3-word question and a 500-word explanation are both "one turn". Write a `trim_to_token_budget(conversation, max_tokens)` that keeps as many recent messages as fit within a token budget.

Hint: a rough token estimate is `len(text) // 4`. For production, use `tiktoken`.

In [ ]:
def trim_to_token_budget(conversation, max_tokens=2000):
    """Keep as many recent messages as fit within a token budget.
    
    Args:
        conversation: list of message dicts
        max_tokens: approximate token budget for the kept messages
    
    Returns:
        list of messages (most recent first) that fit within budget
    """
    # TODO: iterate from the end of conversation, accumulating token estimates
    # Stop when adding the next message would exceed max_tokens
    # Return the kept messages in original order
    pass


# Test it with full_history from section 3
# trimmed = trim_to_token_budget(full_history, max_tokens=500)
# print(f"Kept {len(trimmed)} messages within ~500 token budget")
# for m in trimmed:
#     est = len(m["content"]) // 4
#     print(f"  [{m['role']:>9}] ~{est:>3} tokens: {m['content'][:60]}...")

---
## CLI scripts

For the full interactive experience (try these in your terminal):

```bash
uv run python labs/04_managing_context/1_stateless_agent.py
uv run python labs/04_managing_context/2_stateful_agent.py
uv run python labs/04_managing_context/3_agent_with_memory.py
uv run python labs/04_managing_context/4_agent_with_long_term_memory.py
```

---
## Takeaways

- **LLMs are stateless.** Context is your responsibility — you send it, or it doesn't exist.
- **Full history doesn't scale.** Cost and latency grow quadratically with conversation length.
- **Memory is lossy compression.** You choose what to keep and what to lose. The summarizer decides what matters.
- **Short-term vs long-term.** Short-term resets with the conversation; long-term persists across sessions and requires user identity.
- **Design for your constraints.** Window size, summary format, persistence strategy — all tradeoffs with no universal right answer.